# 04 — Full-parameter SFT design lab: why this is not a one-GPU exercise

This notebook is deliberately honest about the word **full**: all Nemotron 3.5 Lightning parameters receive gradients. It is not QLoRA, not a full-rank adapter, and not selective layer unfreezing.

A single 80 GB GPU can hold the ~60 GB BF16 weights for inference or tight PEFT, but not weights plus full gradients, FP32 optimizer state, activations, and checkpoint buffers. The runnable workshop gate is 8 GPUs / 600 GiB aggregate VRAM. NVIDIA's currently verified 4K H100 reference uses 16×H100 80 GB (TP2, EP8).

Most Brev workshop instances expose one GPU, so this notebook is deliberately design-only: it estimates memory, inspects the required topology, and prints the externally runnable command without launching training. The guarded `scripts/train_full.py` driver remains available for a separately rehearsed multi-GPU environment.

In [ ]:
from pathlib import Path
import json, os, sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
DATA_DIR = ROOT / 'artifacts/data/banking77'
BASELINE_PATH = ROOT / 'artifacts/evaluation/baseline_local_bf16.json'
MEGATRON_BASE = Path('/workspace/storage/checkpoints/lightning35-megatron')
FULL_ROOT = Path('/workspace/storage/checkpoints/banking77-full')
FULL_HF = Path('/workspace/storage/checkpoints/banking77-full-hf')
DESIGN_ONLY = True  # this notebook never launches the expensive job

## 1. Understand the memory boundary

A rough AdamW training budget per parameter is 2 bytes BF16 weight + 2 bytes gradient + 4 bytes FP32 master weight + 8 bytes FP32 moments = 16 bytes before activations, communication buckets, and temporary workspaces. Sparse MoE reduces compute per token, not the total trainable parameter or optimizer-state footprint.

In [ ]:
TOTAL_PARAMETERS = 30_000_000_000
budget = {
    'bf16_weights_gib': TOTAL_PARAMETERS * 2 / 1024**3,
    'bf16_gradients_gib': TOTAL_PARAMETERS * 2 / 1024**3,
    'fp32_master_and_moments_gib': TOTAL_PARAMETERS * 12 / 1024**3,
}
budget['subtotal_before_activations_gib'] = sum(budget.values())
budget

In [ ]:
from nemotron_ft_lab.hardware import inspect_cuda, validate_full_sft_hardware

inventory = inspect_cuda()
print(json.dumps(inventory.as_dict(), indent=2))
try:
    validate_full_sft_hardware(inventory)
    FULL_SFT_CAPABLE = True
    print('Full-SFT hardware gate passed.')
except RuntimeError as exc:
    FULL_SFT_CAPABLE = False
    print('Design-only mode:', exc)

## 2. Inspect the full-SFT topology

The driver starts from NVIDIA's `nemotron_3_5_lightning_sft_config`. On 8 GPUs it uses TP1/EP8; on 16 GPUs it uses the currently verified TP2/EP8 topology. It retains BF16, distributed AdamW, MTP, assistant-only loss, packed 512-token sequences, selective recomputation, and a portable all-to-all MoE dispatcher. No PEFT config is attached.

In [ ]:
TARGET_GPUS = 8
full_cmd = [
    'torchrun', f'--nproc-per-node={TARGET_GPUS}', 'scripts/train_full.py',
    '--megatron-checkpoint', str(MEGATRON_BASE),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(FULL_ROOT),
    '--sequence-length', '512',
    '--global-batch-size', '32',
    '--max-steps', '40',
    '--learning-rate', '5e-6',
]
print('Attached GPUs:', inventory.gpu_count)
print('Minimum handoff command:', ' '.join(full_cmd))
print('Trainable scope: ALL model parameters (no cfg.peft).')

## 3. Handoff for a qualifying allocation

The command below is documentation, not an execution cell. A separate operator should provision and rehearse the qualifying allocation, copy or prepare the checkpoint and frozen dataset, and invoke the hardware-gated driver outside this ordinary Launchable.

In [ ]:
assert DESIGN_ONLY
print('Design-only handoff command (not executed):')
print(' '.join(full_cmd))
print('The driver will still fail below 8 GPUs / 600 GiB aggregate VRAM.')

## 4. Required result contract for any external full-SFT run

A full checkpoint is much larger than an adapter. Export can require substantial host RAM and another ~60+ GB on disk. Any external run must export with `scripts/export_full_checkpoint.py` and compare against Notebook 02's `baseline_local_bf16.json` on the identical official-test IDs.

In [ ]:
required_evidence = {
    'exact_base_checkpoint': str(MEGATRON_BASE),
    'frozen_baseline': str(BASELINE_PATH),
    'frozen_evaluation_ids': str(DATA_DIR / 'manifest.json'),
    'training_topology': 'record TP, EP, DP, GPU type/count, and software revisions',
    'outcome': 'paired exact-accuracy delta with 95% bootstrap CI',
}
print(json.dumps(required_evidence, indent=2))

In [ ]:
print('No training, export, or evaluation was launched by this notebook.')
print('Use Notebook 03 on the normal one-GPU workshop unless a measured adapter ceiling justifies escalation.')

## Decision: adapter or full SFT?

Prefer PEFT when it reaches the held-out target: it is faster, cheaper, easier to version, and preserves the base weights. Consider full SFT only when experiments show a material adapter ceiling, you have enough representative data, and the accuracy gain justifies a multi-GPU training and deployment lifecycle. Keep deterministic validators and human escalation outside either model.